In [12]:
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import cv2
import os
from sklearn.metrics import classification_report, confusion_matrix

In [13]:
# Define paths
train_dir = '/content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/train'
test_dir = '/content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/test'


In [24]:
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from PIL import Image, UnidentifiedImageError # Import Image and UnidentifiedImageError from PIL

def clean_directory(path):
    print(f"Cleaning directory: {path}")
    cleaned_count = 0
    for root, dirs, files in os.walk(path):
        for file in files:
            file_path = os.path.join(root, file)
            remove_file = False
            # Check if the file is a hidden macOS file
            if file.startswith('._'):
                print(f"Removing hidden macOS file: {file_path}")
                remove_file = True
            # Check if the file has a valid image extension
            elif not file.lower().endswith(('.png', '.jpg', '.jpeg')):
                 print(f"Removing file with invalid extension: {file_path}")
                 remove_file = True
            else:
                try:
                    # Try to open and verify the image
                    img = Image.open(file_path)
                    img.verify()
                    img.close()
                except (IOError, SyntaxError, UnidentifiedImageError) as e:
                    print(f"Removing invalid image file: {file_path} - {e}")
                    remove_file = True

            if remove_file:
                try:
                    os.remove(file_path)
                    cleaned_count += 1
                except OSError as e:
                    print(f"Error removing file {file_path}: {e}")
    print(f"Finished cleaning {path}. Removed {cleaned_count} files.")


# Clean the dataset folders
clean_directory(train_dir)
clean_directory(test_dir)

# After cleaning, check if the directories are not empty
if not os.listdir(train_dir):
    print(f"Warning: Training directory {train_dir} is empty after cleaning.")
if not os.listdir(test_dir):
    print(f"Warning: Test directory {test_dir} is empty after cleaning.")

# Check subdirectories within train_dir to see if class folders exist
train_subdirs = [os.path.join(train_dir, d) for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))]
if not train_subdirs:
    print(f"Warning: No class subdirectories found in training directory {train_dir}.")
else:
    print(f"Found class subdirectories in training directory: {train_subdirs}")
    for subdir in train_subdirs:
        if not os.listdir(subdir):
             print(f"Warning: Class subdirectory {subdir} is empty after cleaning.")
        else:
             print(f"Class subdirectory {subdir} contains files.")

# Check subdirectories within test_dir to see if class folders exist
test_subdirs = [os.path.join(test_dir, d) for d in os.listdir(test_dir) if os.path.isdir(os.path.join(test_dir, d))]
if not test_subdirs:
    print(f"Warning: No class subdirectories found in test directory {test_dir}.")
else:
    print(f"Found class subdirectories in test directory: {test_subdirs}")
    for subdir in test_subdirs:
        if not os.listdir(subdir):
             print(f"Warning: Class subdirectory {subdir} is empty after cleaning.")
        else:
             print(f"Class subdirectory {subdir} contains files.")



Cleaning directory: /content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/train
Finished cleaning /content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/train. Removed 0 files.
Cleaning directory: /content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/test
Finished cleaning /content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/test. Removed 0 files.
Found class subdirectories in training directory: ['/content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/train/NORMAL', '/content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/train/PNEUMONIA']
Found class subdirectories in test directory: ['/content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/test/NORMAL', '/content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/test/PNEUMONIA']


In [30]:
# Example (if everything is mixed in train folder)
import os
os.makedirs('/content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/train/NORMAL', exist_ok=True)
os.makedirs('/content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/train/PNEUMONIA', exist_ok=True)

# Move images manually based on naming pattern (adjust if needed)
import shutil

train_path = '/content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/train'
for filename in os.listdir(train_path):
    if "NORMAL" in filename.upper():
        shutil.move(os.path.join(train_path, filename), os.path.join(train_path, 'NORMAL', filename))
    elif "PNEUMONIA" in filename.upper():
        shutil.move(os.path.join(train_path, filename), os.path.join(train_path, 'PNEUMONIA', filename))


Error: Cannot move a directory '/content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/train/NORMAL' into itself '/content/drive/MyDrive/archive (1)/chest_xray/__MACOSX/chest_xray/train/NORMAL/NORMAL'.

In [25]:
# ImageDataGenerator with 20% validation split
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=15,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2
)

# Training data (80%)
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='training'
)

# Validation data (20%)
val_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    subset='validation'
)

# Test data
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(150, 150),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)


Found 0 images belonging to 2 classes.
Found 0 images belonging to 2 classes.
Found 0 images belonging to 2 classes.


In [26]:
# Build CNN model
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(150, 150, 3)),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D(2, 2),

    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.Dense(1, activation='sigmoid')  # Binary classification
])

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)                   │ (None, 148, 148, 32)        │             896 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_12 (MaxPooling2D)      │ (None, 74, 74, 32)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_13 (Conv2D)                   │ (None, 72, 72, 64)          │          18,496 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_13 (MaxPooling2D)      │ (None, 36, 36, 64)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2d_14 (Conv2D)                   │ (None, 34, 34, 128)         │          73,856 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ max_pooling2d_14 (MaxPooling2D)      │ (None, 17, 17, 128)         │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_4 (Flatten)                  │ (None, 36992)               │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_8 (Dense)                      │ (None, 512)                 │      18,940,416 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_9 (Dense)                      │ (None, 1)                   │             513 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 19,034,177 (72.61 MB)

 Trainable params: 19,034,177 (72.61 MB)

 Non-trainable params: 0 (0.00 B)

In [27]:
# Train the model
history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator
)


/usr/local/lib/python3.11/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


ValueError: The PyDataset has length 0